### d) Генерация выборки, ОМП и асимптотические интервалы

In [27]:
import numpy as np
import scipy.stats as st

np.random.seed(42)

def get_pareto_sample(u, param):
    return (1 - u)**(1 / (1 - param))

size_n = 100
conf_level = 0.95
true_theta = 5

uniform_vals = st.uniform(0, 1).rvs(size=size_n)
data_X = get_pareto_sample(uniform_vals, true_theta)
data_X

array([1.12447586, 2.12236357, 1.38983693, 1.25638134, 1.04331821,
       1.04331075, 1.01507215, 1.65335692, 1.25831124, 1.3604459 ,
       1.00521337, 2.40100973, 1.56300052, 1.06148822, 1.05144956,
       1.05195765, 1.09492781, 1.20440053, 1.15186719, 1.08986708,
       1.26692502, 1.03827287, 1.09021931, 1.1208298 , 1.1644323 ,
       1.4688562 , 1.0572635 , 1.19782485, 1.25154174, 1.01196194,
       1.2634338 , 1.04784986, 1.01695817, 2.10311998, 2.32253111,
       1.51147031, 1.09507403, 1.02602727, 1.33400766, 1.15606582,
       1.03307321, 1.18635645, 1.00878681, 1.82231145, 1.07773805,
       1.31201477, 1.09788615, 1.20144827, 1.21872588, 1.05242513,
       2.39456547, 1.45217342, 2.01631907, 1.75599913, 1.25578843,
       1.89147859, 1.02343423, 1.05604804, 1.01163769, 1.10338532,
       1.13092126, 1.08235596, 1.55447748, 1.11662054, 1.08594515,
       1.21604255, 1.03870478, 1.49948378, 1.01955776, 2.9551121 ,
       1.44754779, 1.05694731, 1.00138531, 1.52573185, 1.35903

In [28]:
theta_hat = 1 + 1 / np.mean(np.log(data_X))
real_med = 2 ** (1 / (true_theta - 1))
est_med = 2 ** (1 / (theta_hat - 1))

q1 = st.norm.ppf((1 - conf_level) / 2)
q2 = st.norm.ppf((1 + conf_level) / 2)


med_margin = (est_med * np.log(2)) / (np.sqrt(size_n) * (theta_hat - 1))
ci_med_left = est_med - med_margin * q2
ci_med_right = est_med - med_margin * q1
len_med_asymp = ci_med_right - ci_med_left

print(f"Истинная медиана: {real_med:.4f}, Оценка: {est_med:.4f}")
print(f"Асимпт. интервал (медиана): {ci_med_left:.4f} < x_0.5 < {ci_med_right:.4f}")
print(f"Ширина l = {len_med_asymp:.4f}\n")

theta_margin = 1 / (np.sum(np.log(data_X)) / np.sqrt(size_n))
ci_th_left = theta_hat - theta_margin * q2
ci_th_right = theta_hat - theta_margin * q1
len_th_asymp = ci_th_right - ci_th_left

print(f"Истинная theta: {true_theta}, Оценка ОМП: {theta_hat:.4f}")
print(f"Асимпт. интервал (theta): {ci_th_left:.4f} < theta < {ci_th_right:.4f}")
print(f"Ширина l = {len_th_asymp:.4f}")

Истинная медиана: 1.1892, Оценка: 1.1718
Асимпт. интервал (медиана): 1.1354 < x_0.5 < 1.2082
Ширина l = 0.0728

Истинная theta: 5, Оценка ОМП: 5.3728
Асимпт. интервал (theta): 4.5157 < theta < 6.2298
Ширина l = 1.7141


### t) Бутстраповские доверительные интервалы (непараметрический и параметрический)

In [29]:
B_np = 1000
B_p = 50000
alpha = 1 - conf_level

low_p = alpha / 2 * 100
high_p = (1 - alpha / 2) * 100

idx_matrix = np.random.randint(0, size_n, size=(B_np, size_n))
np_samples = data_X[idx_matrix]

th_stars_np = 1 + 1 / np.mean(np.log(np_samples), axis=1)
deltas_np = th_stars_np - theta_hat

ci_np_left = theta_hat - np.percentile(deltas_np, high_p)
ci_np_right = theta_hat - np.percentile(deltas_np, low_p)
len_np = ci_np_right - ci_np_left

print(f"Непараметрический бутстрап: {ci_np_left:.4f} < theta < {ci_np_right:.4f}")
print(f"Ширина l = {len_np:.4f}\n")

unif_matrix = st.uniform(0, 1).rvs(size=(B_p, size_n))
p_samples = get_pareto_sample(unif_matrix, theta_hat)

th_stars_p = 1 + 1 / np.mean(np.log(p_samples), axis=1)
deltas_p = th_stars_p - theta_hat

ci_p_left = theta_hat - np.percentile(deltas_p, high_p)
ci_p_right = theta_hat - np.percentile(deltas_p, low_p)
len_p = ci_p_right - ci_p_left

print(f"Параметрический бутстрап: {ci_p_left:.4f} < theta < {ci_p_right:.4f}")
print(f"Ширина l = {len_p:.4f}")

Непараметрический бутстрап: 4.3753 < theta < 6.0763
Ширина l = 1.7011

Параметрический бутстрап: 4.3634 < theta < 6.1153
Ширина l = 1.7519


### f) Сравнить полученные доверительные интервалы

In [30]:
intervals = [
    (len_th_asymp, "Асимптотический"),
    (len_np, "Непараметрический бутстрап"),
    (len_p, "Параметрический бутстрап")
]
intervals.sort()

print("Рейтинг интервалов (для theta):")
for idx, (length, name) in enumerate(intervals, 1):
    print(f"{idx}) {name} (l = {length:.4f})")

Рейтинг интервалов (для theta):
1) Непараметрический бутстрап (l = 1.7011)
2) Асимптотический (l = 1.7141)
3) Параметрический бутстрап (l = 1.7519)
